# Dataset 2 - Regresi Konsumsi Energi Appliances
Notebook berisi pipeline ML lengkap untuk dataset Appliances Energy Prediction (target numerik `Appliances` dalam Wh) dengan dua skenario split 80:20 dan 70:30, validasi silang 5-fold, dan hyperparameter tuning RandomizedSearchCV. Plan B menambahkan varian log-transform target dan polynomial features untuk Linear Regression serta perluasan grid SVR.


## 1. Setup dan Konfigurasi
Memuat library numerik, scikit-learn, serta fungsi pembantu untuk menyimpan figure SVG dan tabel CSV.


In [1]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import LinearSVR

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = Path("data/KAG_energydata_complete.csv")
RESULT_DIR = Path("results/regression")
FIGURE_DIR = RESULT_DIR / "figures"
TABLE_DIR = RESULT_DIR / "tables"
MODEL_DIR = RESULT_DIR / "models"
for directory in [FIGURE_DIR, TABLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"


def save_svg(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, format="svg")
    plt.close(fig)
    return path


def show_table(df, filename=None):
    display(df)
    if filename:
        df.to_csv(TABLE_DIR / filename, index=False)


def save_table(df, filename, index=False):
    df.to_csv(TABLE_DIR / filename, index=index)


## 2. Load Data
Mendukung path lokal maupun Google Colab. Memuat `KAG_energydata_complete.csv` lalu menampilkan dimensi dan tipe kolom untuk verifikasi awal.


In [2]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_PATH = Path("/content/drive/MyDrive/UTS_DEEPL_S2/data/KAG_energydata_complete.csv")
    RESULT_DIR = Path("/content/drive/MyDrive/UTS_DEEPL_S2/results/regression")
    FIGURE_DIR = RESULT_DIR / "figures"
    TABLE_DIR  = RESULT_DIR / "tables"
    MODEL_DIR  = RESULT_DIR / "models"
    for directory in [FIGURE_DIR, TABLE_DIR, MODEL_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {DATA_PATH}")
print(f"File ditemukan: {DATA_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File ditemukan: /content/drive/MyDrive/UTS_DEEPL_S2/data/KAG_energydata_complete.csv


Memuat dataset dan menampilkan ringkasan.


In [3]:
raw_df = pd.read_csv(DATA_PATH)
print(f"Shape awal: {raw_df.shape}")
display(raw_df.head())
display(raw_df.info())


Shape awal: (19735, 29)


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 non-null  float64
 19  T9  

None

## 3. EDA Awal
Memeriksa missing value, statistik deskriptif target `Appliances`, distribusi histogram tiap fitur sensor IoT, serta boxplot fitur kunci untuk identifikasi outlier sebelum standardisasi.


In [4]:
missing_table = (
    raw_df.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)
missing_table["missing_percent"] = missing_table["missing_count"] / len(raw_df) * 100
show_table(missing_table, "missing_values_raw.csv")

target_summary = raw_df["Appliances"].describe().reset_index()
target_summary.columns = ["metric", "value"]
show_table(target_summary, "target_summary.csv")

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(raw_df["Appliances"], bins=45, kde=True, color="#6baed6", ax=ax)
ax.set_title("Distribusi Konsumsi Energi Appliances")
ax.set_xlabel("Appliances (Wh)")
ax.set_ylabel("Frekuensi")
save_svg(fig, "target_distribution.svg")

numeric_feature_cols = raw_df.select_dtypes(include=np.number).columns.tolist()
plot_cols = numeric_feature_cols
n_cols = 5
n_rows = int(np.ceil(len(plot_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
for ax, col in zip(axes.ravel(), plot_cols):
    sns.histplot(raw_df[col], bins=35, kde=False, ax=ax, color="#9ecae1")
    ax.set_title(f"Distribusi {col}")
for ax in axes.ravel()[len(plot_cols):]:
    ax.axis("off")
fig.tight_layout()
save_svg(fig, "numeric_distributions_raw.svg")

outlier_cols = ["Appliances", "lights", "T1", "RH_1", "T_out", "RH_out", "Windspeed", "Visibility"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), outlier_cols):
    sns.boxplot(data=raw_df, y=col, ax=ax, color="#a1d99b")
    ax.set_title(f"Outlier {col}")
save_svg(fig, "outlier_boxplots_raw.svg")


,feature,missing_count,missing_percent
0,date,0,0.0
1,Appliances,0,0.0
2,lights,0,0.0
3,T1,0,0.0
4,RH_1,0,0.0
5,T2,0,0.0
6,RH_2,0,0.0
7,T3,0,0.0
8,RH_3,0,0.0
9,T4,0,0.0


,metric,value
0,count,19735.000000
1,mean,97.694958
2,std,102.524891
3,min,10.000000
4,25%,50.000000
5,50%,60.000000
6,75%,100.000000
7,max,1080.000000


PosixPath('/content/drive/MyDrive/UTS_DEEPL_S2/results/regression/figures/outlier_boxplots_raw.svg')

## 4. Feature Engineering
Menambahkan fitur waktu (`hour`, `day`, `month`, `weekday`, `is_weekend`) dari kolom `date` dan menghapus `rv2` yang merupakan duplikat acak `rv1`.


In [5]:
def prepare_energy_data(df):
    prepared = df.copy()
    prepared["date"] = pd.to_datetime(prepared["date"])
    prepared["hour"] = prepared["date"].dt.hour
    prepared["day"] = prepared["date"].dt.day
    prepared["month"] = prepared["date"].dt.month
    prepared["weekday"] = prepared["date"].dt.weekday
    prepared["is_weekend"] = prepared["weekday"].isin([5, 6]).astype(int)
    prepared = prepared.drop(columns=["date", "rv2"])
    return prepared


df = prepare_energy_data(raw_df)
print(f"Shape setelah feature engineering: {df.shape}")
display(df.head())

feature_engineering_table = pd.DataFrame(
    {
        "feature": ["hour", "day", "month", "weekday", "is_weekend", "rv2"],
        "action": ["created", "created", "created", "created", "created", "removed"],
    }
)
show_table(feature_engineering_table, "feature_engineering.csv")


Shape setelah feature engineering: (19735, 32)


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,RH_out,Windspeed,Visibility,Tdewpoint,rv1,hour,day,month,weekday,is_weekend
0,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,...,92.0,7.000000,63.000000,5.3,13.275433,17,11,1,0,0
1,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,...,92.0,6.666667,59.166667,5.2,18.606195,17,11,1,0,0
2,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,...,92.0,6.333333,55.333333,5.1,28.642668,17,11,1,0,0
3,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,...,92.0,6.000000,51.500000,5.0,45.410389,17,11,1,0,0
4,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,...,92.0,5.666667,47.666667,4.9,10.084097,17,11,1,0,0


,feature,action
0,hour,created
1,day,created
2,month,created
3,weekday,created
4,is_weekend,created
5,rv2,removed


## 5. Korelasi dan Feature Selection
Membuat heatmap korelasi seluruh fitur, lalu menghapus pasangan dengan korelasi absolut > 0.95 (mis. `T_out` yang redundan terhadap fitur lain).


In [6]:
fig, ax = plt.subplots(figsize=(14, 11))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2, ax=ax)
ax.set_title("Heatmap Korelasi Dataset Energy")
save_svg(fig, "correlation_heatmap.svg")

X_all = df.drop(columns=["Appliances"])
y = df["Appliances"]

corr_abs = X_all.corr(numeric_only=True).abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
redundant_features = [col for col in upper.columns if any(upper[col] > 0.95)]
X = X_all.drop(columns=redundant_features)

feature_selection_table = pd.DataFrame(
    {
        "removed_feature": redundant_features if redundant_features else ["none"],
        "reason": (
            ["correlation > 0.95"] * len(redundant_features)
            if redundant_features
            else ["no redundant feature above threshold"]
        ),
    }
)
show_table(feature_selection_table, "feature_selection.csv")
print(f"Jumlah fitur akhir: {X.shape[1]}")
print(list(X.columns))


,removed_feature,reason
0,T_out,correlation > 0.95


Jumlah fitur akhir: 30
['lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'hour', 'day', 'month', 'weekday', 'is_weekend']


## 6. Perbandingan Sebelum dan Sesudah Feature Engineering
Mengukur dampak FE dengan validasi silang 5-fold pada Linear Regression untuk dataset mentah dan dataset hasil FE; metrik MAE, MAPE, RMSE, dan R^2 dicatat.


In [7]:
compare_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
compare_model = Pipeline(
    [("scaler", StandardScaler()), ("model", LinearRegression())]
)
compare_scoring = {
    "mae": "neg_mean_absolute_error",
    "mape": "neg_mean_absolute_percentage_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

raw_X_compare = raw_df.drop(columns=["date", "Appliances"])
raw_y_compare = raw_df["Appliances"]

feature_engineering_rows = []
for stage, features, target in [
    ("before_feature_engineering", raw_X_compare, raw_y_compare),
    ("after_feature_engineering", X, y),
]:
    cv_result = cross_validate(
        compare_model, features, target, scoring=compare_scoring,
        cv=compare_cv, n_jobs=1, error_score="raise",
    )
    feature_engineering_rows.append(
        {
            "stage": stage,
            "rows": len(features),
            "features": features.shape[1],
            "cv_mae": -cv_result["test_mae"].mean(),
            "cv_mape": -cv_result["test_mape"].mean() * 100,
            "cv_rmse": -cv_result["test_rmse"].mean(),
            "cv_r2": cv_result["test_r2"].mean(),
        }
    )

feature_engineering_comparison = pd.DataFrame(feature_engineering_rows)
show_table(feature_engineering_comparison, "feature_engineering_comparison.csv")

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=feature_engineering_comparison, x="stage", y="cv_r2", ax=ax, palette="Set2")
ax.set_title("Perbandingan CV R2 Sebelum dan Sesudah Feature Engineering")
ax.set_xlabel("Tahap")
ax.set_ylabel("CV R2 Linear Regression")
ax.tick_params(axis="x", rotation=15)
save_svg(fig, "feature_engineering_comparison.svg")


,stage,rows,features,cv_mae,cv_mape,cv_rmse,cv_r2
0,before_feature_engineering,19735,27,52.958937,61.015376,93.800231,0.162397
1,after_feature_engineering,19735,30,52.829802,60.842740,93.595649,0.166031


PosixPath('/content/drive/MyDrive/UTS_DEEPL_S2/results/regression/figures/feature_engineering_comparison.svg')

## 7. Baseline Model dan Varian Linear Regression
Plan B menjalankan beberapa varian Linear Regression untuk mengeksplorasi non-linearitas dataset: (a) plain, (b) log1p target via `TransformedTargetRegressor`, (c) Polynomial degree-2 interaction-only, (d) Polynomial + log1p target. SVR menggunakan kernel RBF Nystroem + LinearSVR. Semua dievaluasi pada split 80:20 dan 70:30 dengan KFold k=5.


In [8]:
def make_linreg_plain():
    return Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])


def make_linreg_log1p():
    base = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
    return TransformedTargetRegressor(regressor=base, func=np.log1p, inverse_func=np.expm1)


def make_linreg_poly2():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ("model", LinearRegression()),
    ])


def make_linreg_poly2_log1p():
    base = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ("model", LinearRegression()),
    ])
    return TransformedTargetRegressor(regressor=base, func=np.log1p, inverse_func=np.expm1)


def make_svr_baseline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("kernel", Nystroem(kernel="rbf", gamma=0.08, n_components=250, random_state=RANDOM_STATE)),
        ("model", LinearSVR(C=1.0, epsilon=0.1, random_state=RANDOM_STATE, max_iter=10000, dual="auto")),
    ])


def get_regression_models():
    return {
        "Linear Regression": make_linreg_plain(),
        "Linear Regression (log1p)": make_linreg_log1p(),
        "Linear Regression (poly2)": make_linreg_poly2(),
        "Linear Regression (poly2 log1p)": make_linreg_poly2_log1p(),
        "SVR": make_svr_baseline(),
    }


def regression_scores(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "mae": mean_absolute_error(y_test, y_pred),
        "mape": mean_absolute_percentage_error(y_test, y_pred) * 100,
        "rmse": root_mean_squared_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred),
    }


splits = {"80:20": 0.20, "70:30": 0.30}
scoring = {
    "mae": "neg_mean_absolute_error",
    "mape": "neg_mean_absolute_percentage_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

baseline_rows = []
split_data = {}

for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE,
    )
    split_data[split_name] = (X_train, X_test, y_train, y_test)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for model_name, model in get_regression_models().items():
        cv_result = cross_validate(
            model, X_train, y_train, scoring=scoring, cv=cv, n_jobs=1, error_score="raise",
        )
        fitted_model = clone(model).fit(X_train, y_train)
        test_metrics = regression_scores(fitted_model, X_test, y_test)
        baseline_rows.append(
            {
                "split": split_name,
                "model": model_name,
                "cv_mae": -cv_result["test_mae"].mean(),
                "cv_mape": -cv_result["test_mape"].mean() * 100,
                "cv_rmse": -cv_result["test_rmse"].mean(),
                "cv_r2": cv_result["test_r2"].mean(),
                "test_mae": test_metrics["mae"],
                "test_mape": test_metrics["mape"],
                "test_rmse": test_metrics["rmse"],
                "test_r2": test_metrics["r2"],
            }
        )

baseline_results = pd.DataFrame(baseline_rows).sort_values(["split", "cv_r2"], ascending=[True, False])
show_table(baseline_results, "baseline_results.csv")


,split,model,cv_mae,cv_mape,cv_rmse,cv_r2,test_mae,test_mape,test_rmse,test_r2
7,70:30,Linear Regression (poly2),50.715847,57.952764,86.484873,0.291894,50.175722,57.587820,86.040722,0.281444
8,70:30,Linear Regression (poly2 log1p),40.852506,35.579600,88.458068,0.259305,41.057210,36.384192,88.925448,0.232454
5,70:30,Linear Regression,53.011262,60.797085,94.108844,0.161469,52.903402,62.150488,92.583448,0.168008
6,70:30,Linear Regression (log1p),44.791240,39.107327,97.032173,0.109017,44.877964,40.107184,95.857910,0.108116
9,70:30,SVR,44.692568,32.974677,103.622962,-0.015623,44.305714,33.367499,102.066391,-0.011155
2,80:20,Linear Regression (poly2),50.599214,57.825899,86.370297,0.298186,49.625635,57.901779,84.339353,0.289191
3,80:20,Linear Regression (poly2 log1p),40.919851,35.586366,88.738207,0.259313,40.204612,36.248874,86.934981,0.244766
0,80:20,Linear Regression,53.185092,61.067919,94.289550,0.163800,52.552329,62.812846,91.172571,0.169345
1,80:20,Linear Regression (log1p),44.966988,39.254781,97.258630,0.110433,44.333038,40.398007,94.352725,0.110387
4,80:20,SVR,44.862793,33.106028,103.883027,-0.014896,43.173648,32.841307,100.031392,0.000081


## 8. Visualisasi Baseline
Membandingkan CV R^2 dan test RMSE antar varian model dalam dua split untuk memilih kandidat tuning.


In [9]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=baseline_results, x="model", y="cv_r2", hue="split", ax=ax, palette="Set2")
ax.set_title("Perbandingan CV R2 Baseline")
ax.set_xlabel("Model")
ax.set_ylabel("CV R2")
ax.tick_params(axis="x", rotation=25)
save_svg(fig, "baseline_cv_r2_comparison.svg")

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=baseline_results, x="model", y="test_rmse", hue="split", ax=ax, palette="Set3")
ax.set_title("Perbandingan Test RMSE Baseline")
ax.set_xlabel("Model")
ax.set_ylabel("Test RMSE")
ax.tick_params(axis="x", rotation=25)
save_svg(fig, "baseline_test_rmse_comparison.svg")


PosixPath('/content/drive/MyDrive/UTS_DEEPL_S2/results/regression/figures/baseline_test_rmse_comparison.svg')

## 9. Hyperparameter Tuning
Menjalankan RandomizedSearchCV (n_iter=80, scoring=R^2) pada SVR dengan grid yang diperluas (n_components 300-800, gamma 0.005-0.5, C 0.5-50, epsilon 0.05-1.0). Sebagai pembanding adil, varian Linear Regression terbaik dari baseline juga dituning ringan (fit_intercept). Best params kedua kandidat dicatat.


In [10]:
overall_model_score = (
    baseline_results.groupby("model", as_index=False)["cv_r2"]
    .mean()
    .sort_values("cv_r2", ascending=False)
)
print("Ranking varian model berdasar rata-rata CV R2 dua split:")
display(overall_model_score)
overall_model_score.to_csv(TABLE_DIR / "overall_model_score.csv", index=False)

linreg_variants = [
    "Linear Regression",
    "Linear Regression (log1p)",
    "Linear Regression (poly2)",
    "Linear Regression (poly2 log1p)",
]
best_linreg = overall_model_score[overall_model_score["model"].isin(linreg_variants)].iloc[0]["model"]
print(f"Varian Linear Regression terbaik: {best_linreg}")
tuning_candidates = [best_linreg, "SVR"]


def tuning_space(model_name):
    if model_name in linreg_variants:
        estimator = get_regression_models()[model_name]
        if isinstance(estimator, TransformedTargetRegressor):
            params = {"regressor__model__fit_intercept": [True, False]}
        else:
            params = {"model__fit_intercept": [True, False]}
    elif model_name == "SVR":
        estimator = get_regression_models()[model_name]
        params = {
            "kernel__gamma": loguniform(5e-3, 5e-1),
            "kernel__n_components": randint(300, 801),
            "model__C": loguniform(0.5, 50.0),
            "model__epsilon": uniform(0.05, 0.95),
        }
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return estimator, params


N_ITER_SVR = 80
N_ITER_LIN = 4  # tiny grid; n_iter only used as cap
tuning_rows = []
best_estimators = {}

for split_name, (X_train, X_test, y_train, y_test) in split_data.items():
    candidate_results = []
    for cand_name in tuning_candidates:
        estimator, params = tuning_space(cand_name)
        n_iter = N_ITER_SVR if cand_name == "SVR" else N_ITER_LIN
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=params,
            n_iter=n_iter,
            scoring="r2",
            cv=5,
            random_state=RANDOM_STATE,
            n_jobs=1,
            refit=True,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        tuned_model = search.best_estimator_
        tuned_metrics = regression_scores(tuned_model, X_test, y_test)
        baseline_row = baseline_results[
            (baseline_results["split"] == split_name) & (baseline_results["model"] == cand_name)
        ].iloc[0]
        candidate_results.append({
            "split": split_name,
            "model": cand_name,
            "n_iter": n_iter,
            "baseline_cv_r2": baseline_row["cv_r2"],
            "baseline_test_r2": baseline_row["test_r2"],
            "baseline_test_rmse": baseline_row["test_rmse"],
            "baseline_test_mape": baseline_row["test_mape"],
            "tuned_best_cv_r2": search.best_score_,
            "tuned_test_r2": tuned_metrics["r2"],
            "tuned_test_mae": tuned_metrics["mae"],
            "tuned_test_mape": tuned_metrics["mape"],
            "tuned_test_rmse": tuned_metrics["rmse"],
            "best_params": json.dumps(
                {k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()},
                default=str,
            ),
            "fitted_estimator": tuned_model,
        })
    candidate_results.sort(key=lambda r: r["tuned_test_r2"], reverse=True)
    winner = candidate_results[0]
    best_estimators[split_name] = winner["fitted_estimator"]
    for r in candidate_results:
        row = {k: v for k, v in r.items() if k != "fitted_estimator"}
        row["selected_for_split"] = (r["model"] == winner["model"])
        tuning_rows.append(row)

tuning_results = pd.DataFrame(tuning_rows)
show_table(tuning_results, "tuning_results.csv")
joblib.dump(best_estimators, MODEL_DIR / "best_regression_estimators.joblib")
print("\nPemenang per split (berdasar test R2):")
for split_name, est in best_estimators.items():
    name = type(est).__name__
    print(f"  {split_name}: {name}")


Ranking varian model berdasar rata-rata CV R2 dua split:


,model,cv_r2
3,Linear Regression (poly2),0.295040
2,Linear Regression (poly2 log1p),0.259309
0,Linear Regression,0.162635
1,Linear Regression (log1p),0.109725
4,SVR,-0.015260


Varian Linear Regression terbaik: Linear Regression (poly2)


,split,model,n_iter,baseline_cv_r2,baseline_test_r2,baseline_test_rmse,baseline_test_mape,tuned_best_cv_r2,tuned_test_r2,tuned_test_mae,tuned_test_mape,tuned_test_rmse,best_params,selected_for_split
0,80:20,Linear Regression (poly2),4,0.298186,0.289191,84.339353,57.901779,0.297110,0.289191,49.625635,57.901779,84.339353,"{""model__fit_intercept"": true}",True
1,80:20,SVR,80,-0.014896,0.000081,100.031392,32.841307,0.150997,0.159512,39.246854,31.021325,91.710612,"{""kernel__gamma"": 0.06691464038281307, ""kernel...",False
2,70:30,Linear Regression (poly2),4,0.291894,0.281444,86.040722,57.587820,0.291420,0.281444,50.175722,57.587820,86.040722,"{""model__fit_intercept"": true}",True
3,70:30,SVR,80,-0.015623,-0.011155,102.066391,33.367499,0.145711,0.152625,39.874214,30.773513,93.435438,"{""kernel__gamma"": 0.06691464038281307, ""kernel...",False



Pemenang per split (berdasar test R2):
  80:20: Pipeline
  70:30: Pipeline


## 10. Error Plot, Prediksi, dan Perbandingan Tuning
Menyimpan prediksi (y_true, y_pred, residual) per split sebagai CSV agar laporan dapat menghitung distribusi error secara langsung. Plot perbandingan R^2 sebelum dan sesudah tuning, scatter actual vs predicted, dan residual plot disimpan dalam SVG.


In [11]:
selected_rows = tuning_results[tuning_results["selected_for_split"] == True]
comparison_rows = []
for _, row in selected_rows.iterrows():
    comparison_rows.extend([
        {"split": row["split"], "stage": "baseline", "model": row["model"], "r2": row["baseline_test_r2"], "rmse": row["baseline_test_rmse"]},
        {"split": row["split"], "stage": "tuned",    "model": row["model"], "r2": row["tuned_test_r2"],    "rmse": row["tuned_test_rmse"]},
    ])
comparison_df = pd.DataFrame(comparison_rows)
save_table(comparison_df, "tuning_r2_comparison.csv")

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=comparison_df, x="split", y="r2", hue="stage", ax=ax, palette="Set2")
ax.set_title("Perbandingan Test R2 Sebelum dan Sesudah Tuning")
ax.set_xlabel("Split")
ax.set_ylabel("Test R2")
save_svg(fig, "tuning_r2_comparison.svg")

for split_name, tuned_model in best_estimators.items():
    _, X_test, _, y_test = split_data[split_name]
    y_pred = tuned_model.predict(X_test)
    residuals = y_test.values - y_pred

    pred_df = pd.DataFrame({
        "y_true": y_test.values,
        "y_pred": y_pred,
        "residual": residuals,
    })
    pred_df.to_csv(TABLE_DIR / f"predictions_tuned_{split_name.replace(':', '_')}.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(x=y_test, y=y_pred, s=18, alpha=0.45, ax=ax)
    min_value = float(min(y_test.min(), y_pred.min()))
    max_value = float(max(y_test.max(), y_pred.max()))
    ax.plot([min_value, max_value], [min_value, max_value], color="black", linewidth=1)
    ax.set_title(f"Actual vs Predicted - Split {split_name}")
    ax.set_xlabel("Actual Appliances")
    ax.set_ylabel("Predicted Appliances")
    save_svg(fig, f"actual_vs_predicted_tuned_{split_name.replace(':', '_')}.svg")

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(x=y_pred, y=residuals, s=18, alpha=0.45, ax=ax)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Residual Plot - Split {split_name}")
    ax.set_xlabel("Predicted Appliances")
    ax.set_ylabel("Residual")
    save_svg(fig, f"residual_plot_tuned_{split_name.replace(':', '_')}.svg")
